In [59]:
# Standard library
import gc
import warnings
from pathlib import Path
import pickle

# Core data science
import numpy as np
import scipy
import pandas as pd
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow.keras.models import load_model

warnings.filterwarnings("ignore")

DATA_DIR = Path("/share/crsp/lab/pkaiser/ddlin/single-cell-multimodal-ml/data")
RAW_DIR = DATA_DIR.joinpath("raw")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FP_CITE_TRAIN_INPUTS  = RAW_DIR.joinpath("train_cite_inputs.h5")
FP_CITE_TRAIN_TARGETS = RAW_DIR.joinpath("train_cite_targets.h5")
FP_CITE_TEST_INPUTS   = RAW_DIR.joinpath("test_cite_inputs.h5")

FP_IMPORTANT_COLS = PROCESSED_DIR.joinpath("important_cols.txt")
FP_CONSTANT_COLS = PROCESSED_DIR.joinpath("constant_cols.txt")


FP_EVALUATION_IDS     = RAW_DIR.joinpath("evaluation_ids.csv")
FP_CELL_METADATA      = RAW_DIR.joinpath("metadata.csv")

# Verbosity
VERBOSE = 0

# Check GPU availability
if tf.config.list_physical_devices('GPU'):
    print("GPU is available")
else:
    print("GPU is not available, using CPU")

GPU is not available, using CPU


## CITEseq Test prediction

In [48]:
#  Read in the metadata
metadata_df = pd.read_csv(DATA_DIR.joinpath("raw", "metadata.csv"),index_col='cell_id')
metadata_df = metadata_df[metadata_df.technology == "citeseq"]
metadata_df.head()

,day,donor,cell_type,technology
cell_id,,,,
c2150f55becb,2,27678,HSC,citeseq
65b7edf8a4da,2,27678,HSC,citeseq
c1b26cb1057b,2,27678,EryP,citeseq
917168fa6f83,2,27678,NeuP,citeseq
2b29feeca86d,2,27678,EryP,citeseq


In [47]:
# Load the important cols (cell surface protein) and constant cols (cols with no info)
with open(FP_IMPORTANT_COLS, "r") as f:
    important_cols = [line.strip() for line in f]

with open(FP_CONSTANT_COLS, 'r') as s:
    constant_cols = [line.strip() for line in s]

print("First 5 important columns:", important_cols[:5])
print("First 5 constant columns:", constant_cols[:5])

First 5 important columns: ['ENSG00000135218_CD36', 'ENSG00000010278_CD9', 'ENSG00000204287_HLA-DRA', 'ENSG00000117091_CD48', 'ENSG00000004468_CD38']
First 5 constant columns: ['ENSG00000003137_CYP26B1', 'ENSG00000004848_ARX', 'ENSG00000006606_CCL26', 'ENSG00000010379_SLC6A13', 'ENSG00000010932_FMO1']


In [ ]:
# Generate X0t from important cols and standardize it
X = pd.read_hdf(FP_CITE_TRAIN_INPUTS).drop(columns = constant_cols)
cell_index = X.index
meta = metadata_df.reindex(cell_index)
X0 = X[important_cols].values

del X
gc.collect()

# Read test and convert to sparse matrix
Xt = pd.read_hdf(FP_CITE_TEST_INPUTS).drop(columns = constant_cols)
cell_index_test = Xt.index
meta_test = metadata_df.reindex(cell_index_test)
X0t = Xt[important_cols].values

del Xt
gc.collect()

st = StandardScaler()
X0 = st.fit_transform(X0)
X0t = st.transform(X0t)

del X0
gc.collect()

print(f'X0 shape {X0.shape} X0t shape {X0t.shape}')

X0 shape (70988, 84) X0t shape (48663, 84)


In [ ]:
# Load Xt with 512 PCs
CITE_TEST_PKL  = PROCESSED_DIR / "test_Citeseq_truncated_512.pkl"

with open(CITE_TEST_PKL, "rb") as f:
    Xt = pickle.load(f)

In [57]:
Xt_combined = np.hstack((Xt[:,:75],X0t))
Xt_combined.shape

(48663, 159)

In [64]:
Y = pd.read_hdf(FP_CITE_TRAIN_TARGETS)
# Normalize Y by subtracting the mean and dividing by the standard deviation for each cell
# This is important for training stability and performance
Y = Y.values
Y -= Y.mean(axis=1).reshape(-1, 1)
Y /= Y.std(axis=1).reshape(-1, 1)
Y.shape

(70988, 140)

In [58]:
N_SPLITS = 3

def negative_correlation_loss(y_true, y_pred):
    """
    Custom Keras loss function to compute the negative mean Pearson correlation coefficient.

    Parameters:
    - y_true: Ground truth tensor of shape (batch_size, n_targets).
    - y_pred: Predicted tensor of shape (batch_size, n_targets).

    Returns:
    - Tensor: Negative mean Pearson correlation coefficient (to be minimized).
    """
    # Compute mean of predictions across features (axis=1) for each sample
    my = K.mean(tf.convert_to_tensor(y_pred), axis=1)
    
    # Reshape mean to (batch_size, 1) and tile to match y_true shape (batch_size, n_targets)
    my = tf.tile(tf.expand_dims(my, axis=1), (1, y_true.shape[1]))
    
    # Center predictions by subtracting the mean (y_pred - mean(y_pred))
    ym = y_pred - my
    
    # Numerator: Sum of element-wise product of true and centered predicted values
    r_num = K.sum(tf.multiply(y_true, ym), axis=1)
    
    # Denominator: Product of standard deviation of predictions and sqrt(n_targets)
    r_den = tf.sqrt(K.sum(K.square(ym), axis=1) * float(y_true.shape[-1]))
    
    # Compute mean Pearson correlation coefficient across the batch
    r = tf.reduce_mean(r_num / r_den)
    
    # Return negative correlation (to minimize as a loss)
    return -r

In [65]:
test_pred = np.zeros((len(Xt), 140), dtype=np.float32)
for fold in range(N_SPLITS):
    print(f"Predicting with fold {fold}")
    model_path = DATA_DIR.joinpath("models", "citeseq", "submissions")
    model = load_model(model_path.joinpath(f"model_{fold}.keras"),
                       custom_objects={'negative_correlation_loss': negative_correlation_loss})
    test_pred += model.predict(Xt_combined)

# Copy the targets for the data leak but useless since the change in the public LB...
test_pred[:7476] = Y[:7476]

Predicting with fold 0
1521/1521 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Predicting with fold 1
1521/1521 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step
Predicting with fold 2
1521/1521 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step


In [73]:
# Read in the submission 
eval_ids = pd.read_parquet(DATA_DIR.joinpath("processed", "evaluation.parquet"))
eval_ids.cell_id = eval_ids.cell_id.astype(pd.CategoricalDtype())
eval_ids.gene_id = eval_ids.gene_id.astype(pd.CategoricalDtype())

submission = pd.Series(name='target',
                       index=pd.MultiIndex.from_frame(eval_ids), 
                       dtype=np.float32)

# Load the submission with protein expression preds
submission.iloc[:len(test_pred.ravel())] = test_pred.ravel()

submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86               0.094605
1         c2150f55becb  CD274             -0.162362
2         c2150f55becb  CD270             -0.405332
3         c2150f55becb  CD155             -0.302582
4         c2150f55becb  CD112              1.114355
                                             ...   
65744175  2c53aa67933d  ENSG00000134419         NaN
65744176  2c53aa67933d  ENSG00000186862         NaN
65744177  2c53aa67933d  ENSG00000170959         NaN
65744178  2c53aa67933d  ENSG00000107874         NaN
65744179  2c53aa67933d  ENSG00000166012         NaN
Name: target, Length: 65744180, dtype: float32

## Test predictions for Multiome

In [14]:
FILE_PATHS = {
    "train_inputs_embeddings": PROCESSED_DIR / "train_multi_inputs_embeddings_512.pkl",
    "test_inputs_embeddings": PROCESSED_DIR / "test_multi_inputs_embeddings_512.pkl",
    "train_targets_embeddings": PROCESSED_DIR / "train_multi_targets_embeddings_512.pkl",
    "svd_inputs_model": PROCESSED_DIR / "svd_inputs_model_512.pkl",
    "svd_targets_model": PROCESSED_DIR / "svd_targets_model_512.pkl",
}

# Function to load pickled file and print shape (if applicable)
def load_pickle(file_path, name):
    with open(file_path, "rb") as f:
        data = pickle.load(f)
    if hasattr(data, "shape"):
        print(f"Loaded {name}: {data.shape}")
    else:
        print(f"Loaded {name}")
    return data

# Load all data
data = {
    key: load_pickle(path, key) for key, path in FILE_PATHS.items() if path.suffix == ".pkl"
}

# Assign to variables for clarity
X = data["train_inputs_embeddings"]
Xt = data["test_inputs_embeddings"]
Y = data["train_targets_embeddings"]
svd_inputs = data["svd_inputs_model"]
svd_targets = data["svd_targets_model"]

Loaded train_inputs_embeddings: (105942, 512)
Loaded test_inputs_embeddings: (55935, 512)
Loaded train_targets_embeddings: (105942, 512)
Loaded svd_inputs_model
Loaded svd_targets_model


In [15]:
# Load the test sparce matrix
TEST_INPUT_NPZ = PROCESSED_DIR.joinpath("test_multi_inputs_values.npz")
multi_test_x = scipy.sparse.load_npz(TEST_INPUT_NPZ)

# Apply training PCA to get it ready for prediction
multi_test_x = svd_inputs.transform(multi_test_x)
# multi_test_x = multi_test_x[:,:40]
multi_test_x.shape

(55935, 512)

In [ ]:
eval_ids = pd.read_parquet(DATA_DIR.joinpath("processed", "evaluation.parquet"))
eval_ids.cell_id = eval_ids.cell_id.astype(pd.CategoricalDtype())
eval_ids.gene_id = eval_ids.gene_id.astype(pd.CategoricalDtype())

submission = pd.Series(name='target',
                       index=pd.MultiIndex.from_frame(eval_ids), 
                       dtype=np.float32)
submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86              NaN
1         c2150f55becb  CD274             NaN
2         c2150f55becb  CD270             NaN
3         c2150f55becb  CD155             NaN
4         c2150f55becb  CD112             NaN
                                           ..
65744175  2c53aa67933d  ENSG00000134419   NaN
65744176  2c53aa67933d  ENSG00000186862   NaN
65744177  2c53aa67933d  ENSG00000170959   NaN
65744178  2c53aa67933d  ENSG00000107874   NaN
65744179  2c53aa67933d  ENSG00000166012   NaN
Name: target, Length: 65744180, dtype: float32

In [17]:
# This is the test label dense matrix shape
N_SPLITS = 3
model_dir = DATA_DIR.joinpath("models", "atacseq")
preds = np.zeros((multi_test_x.shape[0], 23418), dtype='float16')

for fold in range(N_SPLITS):
    print(f'fold {fold} prediction')
    model_path = model_dir / f"model_{fold}.keras"
    model = tf.keras.models.load_model(model_path)

    # From the predicted PCs, reconstruct the dense matrix
    preds += (model.predict(multi_test_x)@svd_targets.components_)/N_SPLITS
    # model.predict(multi_test_x)

    gc.collect()

fold 0 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
fold 1 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step
fold 2 prediction
1748/1748 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step


In [33]:
# OK we have prediction results for each cell, and the length is 59935 with  23418
type(preds)

numpy.ndarray

In [38]:
TRAIN_META = np.load(
    PROCESSED_DIR.joinpath("train_multi_targets_metadata.npz"),
    allow_pickle=True
    )


TEST_META = np.load(
    PROCESSED_DIR.joinpath("test_multi_inputs_metadata.npz"),
    allow_pickle=True
    )


y_columns = TRAIN_META["columns"]
test_index = TEST_META["index"]



In [42]:
def print_dict(dict):
    n = 0
    for k, v in dict.items():
        print(k, v)
        if n > 5:
            break
        n += 1
    
cell_dict = dict((k,v) for v,k in enumerate(test_index)) 
assert len(cell_dict)  == len(test_index)

gene_dict = dict((k,v) for v,k in enumerate(y_columns))
assert len(gene_dict) == len(y_columns)

print_dict(cell_dict)
print_dict(gene_dict)

458c2ae2c9b1 0
01a0659b0710 1
028a8bc3f2ba 2
7ec0ca8bb863 3
caa0b0022cdc 4
e0bc46450106 5
632ae0df4dcd 6
ENSG00000121410 0
ENSG00000268895 1
ENSG00000175899 2
ENSG00000245105 3
ENSG00000166535 4
ENSG00000256661 5
ENSG00000184389 6


In [ ]:
eval_ids["cell_id"] = eval_ids["cell_id"].astype(str)
eval_ids["gene_id"] = eval_ids["gene_id"].astype(str)

eval_ids_cell_num = eval_ids["cell_id"].map(cell_dict).fillna(-1).astype(int)
eval_ids_gene_num = eval_ids["gene_id"].map(gene_dict).fillna(-1).astype(int)
valid_multi_rows  = (eval_ids_cell_num >= 0) & (eval_ids_gene_num >= 0)

# Total matching pred
print(valid_multi_rows.sum())   # should now be > 0


58931360


In [76]:
submission.iloc[valid_multi_rows] = preds[eval_ids_cell_num[valid_multi_rows].to_numpy(),
eval_ids_gene_num[valid_multi_rows].to_numpy()]

# del eval_ids_cell_num, eval_ids_gene_num, valid_multi_rows, eval_ids, test_index, y_columns
# gc.collect()

submission

row_id    cell_id       gene_id        
0         c2150f55becb  CD86               0.094605
1         c2150f55becb  CD274             -0.162362
2         c2150f55becb  CD270             -0.405332
3         c2150f55becb  CD155             -0.302582
4         c2150f55becb  CD112              1.114355
                                             ...   
65744175  2c53aa67933d  ENSG00000134419    5.859375
65744176  2c53aa67933d  ENSG00000186862    0.044891
65744177  2c53aa67933d  ENSG00000170959    0.053833
65744178  2c53aa67933d  ENSG00000107874    1.262695
65744179  2c53aa67933d  ENSG00000166012    5.546875
Name: target, Length: 65744180, dtype: float32

In [ ]:
TEMP_SUBMISSION = DATA_DIR.joinpath("temp_submission.pkl")
# submission.to_pickle(TEMP_SUBMISSION)

In [82]:
submission.reset_index(drop=True, inplace=True)
submission.index.name = 'row_id'
submission.name = "target"
submission

row_id
0           0.094605
1          -0.162362
2          -0.405332
3          -0.302582
4           1.114355
              ...   
65744175    5.859375
65744176    0.044891
65744177    0.053833
65744178    1.262695
65744179    5.546875
Name: target, Length: 65744180, dtype: float32

In [84]:
FINAL_SUBMISSION = DATA_DIR.joinpath("final_submission.csv")
submission.to_csv(FINAL_SUBMISSION, index=True)

In [ ]:
# kaggle competitions submit -c open-problems-multimodal -f submission.csv -m "Message"

## Total submission

In [ ]:
submission.reset_index(drop=True, inplace=True)
submission.index.name = 'row_id'

cite_submission = pd.read_csv("submission_lolo_1.csv")
cite_submission = cite_submission.set_index("row_id")
cite_submission = cite_submission["target"]
submission[submission.isnull()] = cite_submission[submission.isnull()]
submission
# == > score 0.812


row_id
0           0.094605
1          -0.162362
2          -0.405332
3          -0.302582
4           1.114355
              ...   
65744175    2.785156
65744176   -0.379150
65744177   -0.375732
65744178    0.167603
65744179    2.480469
Name: target, Length: 65744180, dtype: float32

In [ ]:
sub_ensembling = pd.read_csv('../input/5-5-msci22-ensembling-citeseq/submission.csv')
submission1 = sub_ensembling.copy()
submission1['target'] = 0.4 * submission + 0.6 * sub_ensembling['target']
submission1

,row_id,target
0,0,0.182096
1,1,0.010811
2,2,-0.101281
3,3,1.442052
4,4,2.616124
...,...,...
65744175,65744175,5.422715
65744176,65744176,-0.302524
65744177,65744177,-0.293852
65744178,65744178,0.755234


In [ ]:
submission1.to_csv("submission_lolo_total_ensembling.csv", index = False)